In [ ]:
# LimberCloud runtime setup
import os

from limbercloud import ProjectPaths

FOLDER = os.environ.get('LIMBERCLOUD_RUNTIME_ROOT')
if not FOLDER:
    raise RuntimeError('Set LIMBERCLOUD_RUNTIME_ROOT before running this notebook')
PATHS = ProjectPaths.from_root(FOLDER)


In [ ]:
import json

import numpy
import pyccl
import scipy
from matplotlib import pyplot
from matplotlib.gridspec import GridSpec

In [ ]:
# Path
TAG = 'Y10'
LABEL = 'KERNEL'

INFO_FOLDER = str(PATHS.config)
DATA_FOLDER = str(PATHS.survey_data(TAG))
PLOT_FOLDER = str(PATHS.plot_group(LABEL, TAG))
os.makedirs(PLOT_FOLDER, exist_ok=True)


In [ ]:
# Cosmology
with PATHS.config_file('cosmology').open('r') as file:
    COSMOLOGY_INFO = json.load(file)

COSMOLOGY = pyccl.Cosmology(
    h=COSMOLOGY_INFO['H'],
    w0=COSMOLOGY_INFO['W0'],
    wa=COSMOLOGY_INFO['WA'],
    n_s=COSMOLOGY_INFO['NS'],
    A_s=COSMOLOGY_INFO['AS'],
    m_nu=COSMOLOGY_INFO['M_NU'],
    Neff=COSMOLOGY_INFO['N_EFF'],
    Omega_b=COSMOLOGY_INFO['OMEGA_B'],
    Omega_k=COSMOLOGY_INFO['OMEGA_K'],
    Omega_c=COSMOLOGY_INFO['OMEGA_CDM'],
    mass_split='single', matter_power_spectrum='halofit', transfer_function='boltzmann_camb',
    extra_parameters={'camb': {'kmax': 50, 'lmax': 5000, 'halofit_version': 'mead2020_feedback', 'HMCode_logT_AGN': 7.8}}
)

In [ ]:
# Kernel
def terminal_source(chi, chi1, chi2):
    """Rising terminal hat with its moving lower limit and stable narrow limit."""
    if chi < 0 or chi >= chi2:
        return 0.0
    width = chi2 - chi1
    if chi == 0:
        return width / 2
    a = width / chi2
    if chi <= chi1:
        # Whole support: width/2 - chi*K(a), K=sum a^m/[m(m+1)].
        if a < 0.25:
            kappa = sum(a**m / (m*(m+1)) for m in range(1, 33))
        else:
            kappa = 1 + (1-a)/a * numpy.log1p(-a)
        return width/2 - chi*kappa
    # Partial support: lower source limit equals the evaluation point.
    u = (chi2-chi)/chi2
    if chi1 == 0:
        return chi2*u*u/2
    if u < 0.25:
        tail = sum(u**m / (m*(m-1)) for m in range(3, 35))
    else:
        tail = u-u*u/2 + (1-u)*numpy.log1p(-u)
    return chi2*chi2/width * (a*u*u/2 - (1-a)*tail)

def r_n_k(k, chi, chi_grid):

    grid_size = chi_grid.shape[0] - 1
    if k == grid_size:
        return terminal_source(chi, chi_grid[-2], chi_grid[-1])
    n = numpy.max(numpy.where(chi_grid <= chi))

    if chi == 0:
        if k < n < grid_size:
            r = 0

        elif n == k < grid_size:
            r = chi_grid[n + 1] ** 2 / 2 / (chi_grid[n + 1] - chi_grid[n])

        elif n + 1 == k < grid_size:
            r = (chi_grid[n + 2] - chi_grid[n + 1]) / 2 + (chi_grid[n + 1] ** 2 - 2 * chi_grid[n] * chi_grid[n + 1]) / 2 / (chi_grid[n + 1] - chi_grid[n])

        elif n + 1 < k < grid_size:
            r = (chi_grid[k + 1] - chi_grid[k - 1]) / 2


        else:
            r = 0

    elif 0 < chi < chi_grid.max():
        if k < n:
            r = 0

        elif n == k < grid_size:
            r = (chi_grid[n + 1] ** 2 - chi ** 2) / 2 / (chi_grid[n + 1] - chi_grid[n]) - chi * chi_grid[n + 1] / (chi_grid[n + 1] - chi_grid[n]) * numpy.log(chi_grid[n + 1] / chi)

        elif n + 1 == k < grid_size:
            r = (chi_grid[n + 2] - chi_grid[n + 1]) / 2 - chi * chi_grid[n + 2] / (chi_grid[n + 2] - chi_grid[n + 1]) * numpy.log(chi_grid[n + 2] / chi_grid[n + 1]) + (chi_grid[n + 1] ** 2 - 2 * chi_grid[n] * chi_grid[n + 1] + chi ** 2) / 2 / (chi_grid[n + 1] - chi_grid[n]) + chi * chi_grid[n] / (chi_grid[n + 1] - chi_grid[n]) * numpy.log(chi_grid[n + 1] / chi)

        elif n + 1 < k < grid_size:
            r = (chi_grid[k + 1] - chi_grid[k - 1]) / 2 - chi * chi_grid[k + 1] / (chi_grid[k + 1] - chi_grid[k]) * numpy.log(chi_grid[k + 1] / chi_grid[k]) + chi * chi_grid[k - 1] / (chi_grid[k] - chi_grid[k-1]) * numpy.log(chi_grid[k] / chi_grid[k - 1])


        else:
            r = 0

    else:
        r = 0

    return r

def r_n(chi, chi_grid):

    grid_size = chi_grid.shape[0] - 1
    r = numpy.zeros(grid_size + 1)

    for k in range(grid_size + 1):
        r[k] = r_n_k(k, chi, chi_grid)
    return r

def q_m(chi_data, chi_grid, phi_grid):
    bin_size = phi_grid.shape[0]
    data_size = chi_data.shape[0] - 1
    q = numpy.zeros((bin_size, data_size + 1))

    for n in range(data_size + 1):
        q[:,n] = numpy.sum(r_n(chi_data[n], chi_grid) * phi_grid, axis = 1)
    return q

def q_m_chi(chi_data, chi_grid, phi_m_grid, type):
    phi_m = scipy.interpolate.interp1d(chi_grid, phi_m_grid, kind=type)
    data_size = chi_data.shape[0] - 1
    q = numpy.zeros(data_size + 1)

    for n in range(data_size + 1):
        def function(chi_prime):
            return phi_m(chi_prime) * (chi_prime - chi_data[n]) / chi_prime
        q[n] = (0.0 if chi_data[n] == chi_grid.max() else
                scipy.integrate.fixed_quad(func=function, a=chi_data[n], b=chi_grid.max(), n=500)[0])

    return q

### Final source node: whole and partial support

For the final rising hat, $R_N(u)=(u-\chi_{N-1})/(\chi_N-\chi_{N-1})$,

$$G_N(\chi)=\int_{\max(\chi,\chi_{N-1})}^{\chi_N}
R_N(u)\frac{u-\chi}{u}\,\mathrm{d}u.$$

Below $\chi_{N-1}$ this is the whole-support expression. Inside the last interval it uses the partial source support; at $\chi_N$ it is zero. The terminal node has no descending hat beyond the analysis domain. The `terminal_source` branch completes the same inside/outside rule used for interior nodes, without reading a nonexistent next node.

For stable evaluation write $a=(\chi_N-\chi_{N-1})/\chi_N$ and $v=(\chi_N-\chi)/\chi_N$. The partial expression is

$$G_N=\frac{\chi_N}{a}\left[\frac{av^2}{2}-(1-a)T(v)\right],\qquad
T(v)=v-\frac{v^2}{2}+(1-v)\log(1-v)
=\sum_{m=3}^{\infty}\frac{v^m}{m(m-1)}.$$

The series prevents cancellation when the interval is narrow. The whole-support branch similarly uses $K(a)=\sum_{m=1}^{\infty}a^m/[m(m+1)]$. At the observer, $G_N(0)=(\chi_N-\chi_{N-1})/2$. The following nonzero terminal-basis fixture checks the branch even when a survey distribution has a negligible final sample.


In [ ]:
# Terminal basis: direct moving-limit integration, independent of the formula.
for CHI_GRID_TEST in (numpy.array([0., 1., 2.]),
                      numpy.array([0., 1., 1.+1.e-5]),
                      numpy.array([0., 1.])):
    CHI1, CHI2 = CHI_GRID_TEST[-2:]
    for CHI_TEST in (0., CHI1, (CHI1+CHI2)/2, CHI2):
        if CHI_TEST == CHI2:
            INTEGRAL = 0.0
        else:
            LOWER = max(CHI_TEST, CHI1)
            INTEGRAL = scipy.integrate.quad(
                lambda U: ((U-CHI1)/(CHI2-CHI1))*(U-CHI_TEST)/U,
                LOWER, CHI2, epsabs=1.e-25, epsrel=1.e-10,
            )[0]
        COEFFICIENT = r_n_k(len(CHI_GRID_TEST)-1, CHI_TEST, CHI_GRID_TEST)
        numpy.testing.assert_allclose(COEFFICIENT, INTEGRAL, rtol=2.e-9, atol=1.e-25)
print('Terminal basis: ordinary, narrow and observer support checks passed.')


In [ ]:
# Grid
Z1 = 0.0
Z2 = 3.5
GRID_SIZE = 350
Z_GRID = numpy.linspace(Z1, Z2, GRID_SIZE + 1)

# Source
SOURCE = numpy.load(os.path.join(DATA_FOLDER, 'lsst_source_bins.npy'), allow_pickle=True).item()
SOURCE_REDSHIFT = SOURCE['redshift_range']
SOURCE_BIN_SIZE = len(SOURCE['bins'])

SOURCE_PSI_GRID = numpy.zeros((SOURCE_BIN_SIZE, GRID_SIZE + 1))
for BIN_INDEX in range(SOURCE_BIN_SIZE):
    SOURCE_PSI_GRID[BIN_INDEX, :] = numpy.interp(x=Z_GRID, xp=SOURCE_REDSHIFT, fp=SOURCE['bins'][BIN_INDEX])
SOURCE_PSI_GRID = SOURCE_PSI_GRID / scipy.integrate.trapezoid(x=Z_GRID, y=SOURCE_PSI_GRID, axis=1)[:, numpy.newaxis]

In [ ]:
# Comparison
A_GRID = 1 / (1 + Z_GRID)
CHI_GRID = pyccl.background.comoving_radial_distance(cosmo=COSMOLOGY, a=A_GRID)
SOURCE_PHI_GRID = SOURCE_PSI_GRID * pyccl.background.h_over_h0(cosmo=COSMOLOGY, a=A_GRID) * COSMOLOGY_INFO['H'] * 100000 / scipy.constants.c

CHI_SIZE = 500
Z_DATA = numpy.linspace(Z1, Z2, CHI_SIZE + 1)

A_DATA = 1 / (1 + Z_DATA)
CHI_DATA = pyccl.background.comoving_radial_distance(cosmo=COSMOLOGY, a=A_DATA)

KERNEL_CCL = numpy.zeros((SOURCE_BIN_SIZE, CHI_SIZE + 1))
KERNEL_DATA1 = numpy.zeros((SOURCE_BIN_SIZE, CHI_SIZE + 1))
KERNEL_DATA2 = numpy.zeros((SOURCE_BIN_SIZE, CHI_SIZE + 1))
KERNEL_DATA3 = numpy.zeros((SOURCE_BIN_SIZE, CHI_SIZE + 1))
KERNEL_DATA = 3 * COSMOLOGY_INFO['OMEGA_M'] / 2 * (COSMOLOGY_INFO['H'] * 100000 / scipy.constants.c)**2 / A_DATA * CHI_DATA * q_m(chi_data=CHI_DATA, chi_grid=CHI_GRID, phi_grid=SOURCE_PHI_GRID)

for BIN_INDEX in range(SOURCE_BIN_SIZE):
    KERNEL_DATA1[BIN_INDEX, :] = 3 * COSMOLOGY_INFO['OMEGA_M'] / 2 * (COSMOLOGY_INFO['H'] * 100000 / scipy.constants.c)**2 / A_DATA * CHI_DATA * q_m_chi(chi_data=CHI_DATA, chi_grid=CHI_GRID, phi_m_grid=SOURCE_PHI_GRID[BIN_INDEX, :], type='slinear')
    KERNEL_DATA2[BIN_INDEX, :] = 3 * COSMOLOGY_INFO['OMEGA_M'] / 2 * (COSMOLOGY_INFO['H'] * 100000 / scipy.constants.c)**2 / A_DATA * CHI_DATA * q_m_chi(chi_data=CHI_DATA, chi_grid=CHI_GRID, phi_m_grid=SOURCE_PHI_GRID[BIN_INDEX, :], type='quadratic')
    KERNEL_DATA3[BIN_INDEX, :] = 3 * COSMOLOGY_INFO['OMEGA_M'] / 2 * (COSMOLOGY_INFO['H'] * 100000 / scipy.constants.c)**2 / A_DATA * CHI_DATA * q_m_chi(chi_data=CHI_DATA, chi_grid=CHI_GRID, phi_m_grid=SOURCE_PHI_GRID[BIN_INDEX, :], type='cubic')

    TRACER = pyccl.tracers.WeakLensingTracer(cosmo=COSMOLOGY, dndz=[Z_GRID, SOURCE_PSI_GRID[BIN_INDEX, :]], has_shear=True, ia_bias=None, use_A_ia=False, n_samples=GRID_SIZE + 1)
    KERNEL_CCL[BIN_INDEX, :] = TRACER.get_kernel(chi=CHI_DATA)

KERNEL_RATIO = numpy.divide(KERNEL_DATA, KERNEL_CCL, out=numpy.ones(KERNEL_CCL.shape), where=KERNEL_CCL > 0)
KERNEL_RATIO1 = numpy.divide(KERNEL_DATA1, KERNEL_CCL, out=numpy.ones(KERNEL_CCL.shape), where=KERNEL_CCL > 0)
KERNEL_RATIO2 = numpy.divide(KERNEL_DATA2, KERNEL_CCL, out=numpy.ones(KERNEL_CCL.shape), where=KERNEL_CCL > 0)
KERNEL_RATIO3 = numpy.divide(KERNEL_DATA3, KERNEL_CCL, out=numpy.ones(KERNEL_CCL.shape), where=KERNEL_CCL > 0)

In [ ]:
TEXLIVE_BIN = os.environ.get('LIMBERCLOUD_TEXLIVE_BIN')
if TEXLIVE_BIN:
    os.environ['PATH'] = TEXLIVE_BIN + os.pathsep + os.environ['PATH']
pyplot.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
pyplot.rcParams['pgf.texsystem'] = 'pdflatex'
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.size'] = 30
KERNEL0 = 1e-5

for BIN_INDEX in range(SOURCE_BIN_SIZE):
    FIGURE = pyplot.figure(figsize=(12, 10))
    GRIDSPEC = GridSpec(10, 12, figure=FIGURE, wspace=0.0, hspace=0.0)

    PLOT = FIGURE.add_subplot(GRIDSPEC[0:8, :])

    PLOT.plot(CHI_DATA, KERNEL_CCL[BIN_INDEX, :] / KERNEL0, color='black', linestyle='-', linewidth=2.5, label=r'$\mathtt{CCL}$', rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_DATA1[BIN_INDEX, :] / KERNEL0, color='blue', linestyle='-', linewidth=2.5, label=r'$\mathrm{Numeric \: 1st}$', rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_DATA2[BIN_INDEX, :] / KERNEL0, color='green', linestyle='-', linewidth=2.5, label=r'$\mathrm{Numeric \: 2nd}$', rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_DATA3[BIN_INDEX, :] / KERNEL0, color='purple', linestyle='-', linewidth=2.5, label=r'$\mathrm{Numeric \: 3rd}$', rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_DATA[BIN_INDEX, :] / KERNEL0, color='orange', linestyle='-', linewidth=2.5, label=r'$\mathtt{LimberCloud}$', rasterized=True)

    PLOT.text(x=5000, y=KERNEL_CCL[BIN_INDEX, :].max() / KERNEL0 / 5, s=rf'$m = {BIN_INDEX + 1:.0f}$', fontsize=30)

    PLOT.set_xlim(CHI_DATA.min(), CHI_DATA.max())
    PLOT.legend(loc='upper right', fontsize=25)

    PLOT.set_xticklabels([])
    PLOT.get_yticklabels()[0].set_visible([])
    PLOT.set_ylabel(r'$\mathcal{W}_\kappa^m (\chi) \: [10^{-5} \: \mathrm{Mpc}^{-2}]$')

    PLOT = FIGURE.add_subplot(GRIDSPEC[8:, :])

    PLOT.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1), color='black', linestyle='-', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1) * (1.00 + 0.01), color='grey', linestyle='--', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1) * (1.00 - 0.01), color='grey', linestyle='--', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_RATIO1[BIN_INDEX, :], color='blue', linestyle='-', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_RATIO2[BIN_INDEX, :], color='green', linestyle='-', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_RATIO3[BIN_INDEX, :], color='purple', linestyle='-', linewidth=2.5, rasterized=True)

    PLOT.plot(CHI_DATA, KERNEL_RATIO[BIN_INDEX, :], color='orange', linestyle='-', linewidth=2.5, rasterized=True)

    PLOT.set_xlim(CHI_DATA.min(), CHI_DATA.max())
    PLOT.get_xticklabels()[0].set_visible([])
    PLOT.set_ylim(1.00 - 0.05, 1.00 + 0.05)

    PLOT.set_xlabel(r'$\chi \: [\mathrm{Mpc}]$')
    PLOT.set_ylabel(r'$\mathrm{Ratio}$')

    FIGURE.subplots_adjust(hspace=0.0, wspace=0.0)
    FIGURE.savefig(os.path.join(PLOT_FOLDER, f'KAPPA{BIN_INDEX + 1}.pdf'), bbox_inches = 'tight', dpi=512)

In [ ]:
TEXLIVE_BIN = os.environ.get('LIMBERCLOUD_TEXLIVE_BIN')
if TEXLIVE_BIN:
    os.environ['PATH'] = TEXLIVE_BIN + os.pathsep + os.environ['PATH']
pyplot.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
pyplot.rcParams['pgf.texsystem'] = 'pdflatex'
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.size'] = 22
KERNEL0 = 1e-5

N_ROWS = 2
N_COLUMNS = 3
N_PANELS = N_ROWS * N_COLUMNS

FIGURE = pyplot.figure(figsize=(25, 15))
OUTER_GRID = GridSpec(N_ROWS, N_COLUMNS, figure=FIGURE, wspace=0.10, hspace=0.15)

for BIN_INDEX in range(SOURCE_BIN_SIZE):
    PANEL_ROW = BIN_INDEX // N_COLUMNS
    PANEL_COLUMN = BIN_INDEX % N_COLUMNS

    INNER_GRID = OUTER_GRID[PANEL_ROW, PANEL_COLUMN].subgridspec(10, 12, wspace=0.0, hspace=0.0)

    PLOT_TOP = FIGURE.add_subplot(INNER_GRID[0:8, :])
    PLOT_BOTTOM = FIGURE.add_subplot(INNER_GRID[8:, :])

    PLOT_TOP.plot(CHI_DATA, KERNEL_CCL[BIN_INDEX, :] / KERNEL0, color='black', linestyle='-', linewidth=2.0, label=r'$\mathtt{CCL}$', rasterized=True)
    PLOT_TOP.plot(CHI_DATA, KERNEL_DATA1[BIN_INDEX, :] / KERNEL0, color='blue', linestyle='-', linewidth=2.0, label=r'$\mathrm{Numeric \: 1st}$', rasterized=True)
    PLOT_TOP.plot(CHI_DATA, KERNEL_DATA2[BIN_INDEX, :] / KERNEL0, color='green', linestyle='-', linewidth=2.0, label=r'$\mathrm{Numeric \: 2nd}$', rasterized=True)
    PLOT_TOP.plot(CHI_DATA, KERNEL_DATA3[BIN_INDEX, :] / KERNEL0, color='purple', linestyle='-', linewidth=2.0, label=r'$\mathrm{Numeric \: 3rd}$', rasterized=True)
    PLOT_TOP.plot(CHI_DATA, KERNEL_DATA[BIN_INDEX, :] / KERNEL0, color='orange', linestyle='-', linewidth=2.0, label=r'$\mathtt{LimberCloud}$', rasterized=True)

    PLOT_TOP.text(x=5000, y=KERNEL_CCL[BIN_INDEX, :].max() / KERNEL0 / 5 * 4, s=rf'$a = {BIN_INDEX + 1:.0f}$', fontsize=35)

    PLOT_TOP.set_xticklabels([])
    PLOT_TOP.set_xlim(CHI_DATA.min(), CHI_DATA.max())

    if PANEL_COLUMN == 0:
        PLOT_TOP.set_ylabel(r'$\mathcal{W}_\kappa^a (\chi) \: [10^{-5} \: \mathrm{Mpc}^{-2}]$')
    else:
        PLOT_TOP.set_ylabel('')

    LEGEND_HANDLES, LEGEND_LABELS = PLOT_TOP.get_legend_handles_labels()

    PLOT_BOTTOM.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1), color='black', linestyle='-', linewidth=2.0, rasterized=True)
    PLOT_BOTTOM.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1) * (1.00 + 0.01), color='grey', linestyle='--', linewidth=2.0, rasterized=True)
    PLOT_BOTTOM.plot(CHI_DATA, numpy.ones(CHI_SIZE + 1) * (1.00 - 0.01), color='grey', linestyle='--', linewidth=2.0, rasterized=True)

    PLOT_BOTTOM.plot(CHI_DATA, KERNEL_RATIO1[BIN_INDEX, :], color='blue', linestyle='-', linewidth=2.0, rasterized=True)
    PLOT_BOTTOM.plot(CHI_DATA, KERNEL_RATIO2[BIN_INDEX, :], color='green', linestyle='-', linewidth=2.0, rasterized=True)
    PLOT_BOTTOM.plot(CHI_DATA, KERNEL_RATIO3[BIN_INDEX, :], color='purple', linestyle='-', linewidth=2.0, rasterized=True)
    PLOT_BOTTOM.plot(CHI_DATA, KERNEL_RATIO[BIN_INDEX, :], color='orange', linestyle='-', linewidth=2.0, rasterized=True)

    PLOT_BOTTOM.set_ylim(1.00 - 0.05, 1.00 + 0.05)
    PLOT_BOTTOM.set_xlim(CHI_DATA.min(), CHI_DATA.max())

    PLOT_BOTTOM.set_xlabel(r'$\chi \: [\mathrm{Mpc}]$')
    if PANEL_COLUMN == 0:
        PLOT_BOTTOM.set_ylabel(r'$\mathrm{Ratio}$')
    else:
        PLOT_BOTTOM.set_ylabel('')

LEGEND_AXIS = FIGURE.add_subplot(OUTER_GRID[N_ROWS - 1, N_COLUMNS - 1])
LEGEND_AXIS.axis('off')

LEGEND_HANDLES, LEGEND_LABELS = FIGURE.axes[0].get_legend_handles_labels()
LEGEND_AXIS.legend(LEGEND_HANDLES, LEGEND_LABELS, loc='center', frameon=False, fontsize=35)

FIGURE.subplots_adjust(top=0.92, bottom=0.08, left=0.08, right=0.98)
FIGURE.savefig(os.path.join(PLOT_FOLDER, 'KAPPA.pdf'), bbox_inches='tight', dpi=512)